# Online Fraud Detection - Exploratory Data Analysis & Modeling Walkthrough

Welcome to the end-to-end Machine Learning pipeline for **Online Fraud Detection**.
This notebook demonstrates:
1. **Safe Data Ingestion & Schema Validation**
2. **Exploratory Data Analysis (EDA)** on transaction behavior
3. **Domain Feature Engineering** (balance errors, depletion indicators)
4. **Class Imbalance Strategy** (Stratified split, balanced class weights, scale_pos_weight)
5. **Model Training & Comparison**: Random Forest vs XGBoost
6. **Empirical Evaluation**: Recall, Precision, F1, ROC-AUC, and PR-AUC curves
7. **Feature Importance Interpretation**

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in system path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import get_dataset_path, RAW_DATA_DIR, SAMPLE_FILENAME
from src.data_loader import load_data, get_dataset_summary
from src.preprocessor import build_preprocessing_pipeline, prepare_features_and_target
from src.model_trainer import train_and_evaluate_models
from src.evaluator import build_comparison_dataframe

## 1. Load Dataset & Summary Statistics

In [ ]:
df = load_data()
print(f"Loaded dataset with {len(df):,} transactions.")
df.head()

In [ ]:
summary = get_dataset_summary(df)
print("Executive Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

## 2. Class Imbalance & Transaction Types
In financial fraud, fraud represents a tiny minority (< 2%).
Let's inspect fraud occurrences across different transaction types.

In [ ]:
from src.eda import get_fraud_rate_by_type, plot_class_balance_figure

type_stats = get_fraud_rate_by_type(df)
type_stats

**Key Finding**: Fraud occurs almost exclusively in **TRANSFER** and **CASH_OUT** channels.
Payment and Cash-In transactions have virtually no fraud.

In [ ]:
fig = plot_class_balance_figure(df)
plt.show()

## 3. Domain Feature Engineering
We calculate:
- `errorBalanceOrig = (oldbalanceOrg - amount) - newbalanceOrig`
- `errorBalanceDest = (oldbalanceDest + amount) - newbalanceDest`
- `orig_balance_depleted = (newbalanceOrig == 0 & amount > 0)`
- `amount_to_oldbalance_ratio = amount / (oldbalanceOrg + 1)`

In [ ]:
from src.eda import get_key_behavioral_insights
insights = get_key_behavioral_insights(df)
insights

## 4. Train and Evaluate Random Forest & XGBoost Models
Let's execute the full training workflow with 80/20 stratified split, inverse class weights, and empirical evaluation.

In [ ]:
results = train_and_evaluate_models(save_artifacts=True)

comp_table = build_comparison_dataframe(
    results["random_forest"]["metrics"],
    results["xgboost"]["metrics"],
)
comp_table

## 5. Performance Curves (ROC and Precision-Recall)

In [ ]:
from src.evaluator import plot_roc_curves_figure, plot_pr_curves_figure

fig_roc = plot_roc_curves_figure(
    results["random_forest"]["metrics"],
    results["xgboost"]["metrics"],
)
plt.show()

fig_pr = plot_pr_curves_figure(
    results["random_forest"]["metrics"],
    results["xgboost"]["metrics"],
)
plt.show()

## 6. Live Inference Demonstration
Test a single transaction using the trained inference engine.

In [ ]:
from src.predictor import FraudPredictor

predictor = FraudPredictor()
sample_fraud = {
    "step": 150,
    "type": "TRANSFER",
    "amount": 180000.0,
    "oldbalanceOrg": 180000.0,
    "newbalanceOrig": 0.0,
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,
}
verdict = predictor.predict_single(sample_fraud, model_name="xgboost")
print("Fraud Detection Verdict:")
for k, v in verdict.items():
    print(f"  {k}: {v}")